# 2. Knapsack problem

Lecture: *Week 5 - Mixed Integer Programming*, PDF pages 11–13 (*Knapsack Problem*).

**The problem.** Seven items with values £10, £10, £10, £100, £100, £130, £70 and weights 2, 2, 2, 6, 6, 8, 3 kg.
You can carry 10 kg. Which items do you take to maximise the total value? Each item is taken once or not at all,
so $x_i = 1$ if item $i$ is packed and $0$ if not:

maximise $\sum_{i=1}^{7} v_i x_i$ subject to $\sum_{i=1}^{7} w_i x_i \le 10$, $x_i \in \{0, 1\}$.

In the code, items 1–7 are `x[0]` … `x[6]`.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Value **£170**: item 5 and item 7 (`x[4] = 1`, `x[6] = 1`), 9 kg. Item 4 and item 7 give
the same £170, so another solver may show that choice instead.

*Original script: `5 Source Files/Python/knapsack_lecture.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A *concrete model* is the container for the problem.

In [ ]:
# Create a concrete model
model = ConcreteModel()

One binary (0/1) variable per item: `Binary` = an integer between 0 and 1 (PDF page 10).

In [ ]:
# Define the binary variables: x[i] = 1 if item i+1 is packed, 0 if not
model.x = Var(range(7), within=Binary)

The data from the table on PDF page 12.

In [ ]:
# Define objective function values and weights
value = [10, 10, 10, 100, 100, 130, 70]  # value of items 1-7 (£)
weight = [2, 2, 2, 6, 6, 8, 3]           # weight of items 1-7 (kg)
W = 10                                   # capacity of the knapsack (kg)

The objective: the total value of the packed items, maximised.

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(value[i] * model.x[i] for i in range(7)), sense=maximize)

One constraint: the packed items must not weigh more than $W$.

In [ ]:
# Define the capacity constraint: sum of w_i x_i <= W
model.capacity = Constraint(expr=sum(weight[i] * model.x[i] for i in range(7)) <= W)

Create the HiGHS solver, print the model (`pprint()`: check it against the problem above), solve, and print the results (`display()`).

The HiGHS log appears between the two printouts (in the browser it is printed even without `tee=True`); its *Solving report* should say *Status Optimal*. In `display()`, look at the `Value` column of the variables and at the objective.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')
# Display the model
model.pprint()
# Solve the model
solver.solve(model)
# Display the results
model.display()

## What do you notice?

Item 6 is worth the most (£130). Why is it not in the knapsack?

<details><summary>Answer</summary>

It weighs 8 kg, which leaves only 2 kg: room for one £10 item, £140 in total. Items 5 and 7 weigh 9 kg and are
worth £170. Per kilogram, item 7 is the best (£23.3/kg) and item 6 the worst of the expensive items
(£16.25/kg). A knapsack needs the best *combination* that fits, not the most valuable single item.
</details>